# Custom Multi-Scale CNN Architecture Training
## Facial Emotion Recognition with Feature Fusion

This notebook trains a custom CNN architecture with multi-scale feature fusion on the facial emotion recognition dataset. The architecture combines multiple pathways with different kernel sizes and pooling operations for robust emotion classification.

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# TensorFlow & Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, MaxPooling2D, 
    Flatten, Dense, Dropout, concatenate, Layer
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model

# Sklearn for metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    precision_score, recall_score, f1_score
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

## 2. Load and Preprocess Dataset

In [ ]:
# Path to preprocessed data
data_path = '../data/preprocessed'

# Load preprocessed numpy arrays
X_train = np.load(os.path.join(data_path, 'X_train.npy'))
X_val = np.load(os.path.join(data_path, 'X_val.npy'))
X_test = np.load(os.path.join(data_path, 'X_test.npy'))

y_train = np.load(os.path.join(data_path, 'y_train.npy'))
y_val = np.load(os.path.join(data_path, 'y_val.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy'))

# Load class weights and mapping
with open(os.path.join(data_path, 'class_weights.json'), 'r') as f:
    class_weights_dict = json.load(f)

with open(os.path.join(data_path, 'class_mapping.json'), 'r') as f:
    class_mapping = json.load(f)

# Convert keys to integers
class_weights_dict = {int(k): v for k, v in class_weights_dict.items()}

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Number of classes: {len(class_mapping)}")
print(f"Class mapping: {class_mapping}")
print(f"Class weights: {class_weights_dict}")

In [ ]:
# Normalize pixel values to [0, 1] range
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Ensure data has the right shape (N, H, W, C)
if len(X_train.shape) == 3:
    X_train = np.expand_dims(X_train, axis=-1)
    X_val = np.expand_dims(X_val, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)

# Convert to 3 channels if grayscale
if X_train.shape[-1] == 1:
    X_train = np.concatenate([X_train, X_train, X_train], axis=-1)
    X_val = np.concatenate([X_val, X_val, X_val], axis=-1)
    X_test = np.concatenate([X_test, X_test, X_test], axis=-1)

print(f"Normalized training shape: {X_train.shape}")
print(f"Data range: [{X_train.min():.4f}, {X_train.max():.4f}]")

# Convert labels to one-hot encoding
from tensorflow.keras.utils import to_categorical
num_classes = len(class_mapping)
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print(f"One-hot encoded shapes - Train: {y_train_cat.shape}, Val: {y_val_cat.shape}, Test: {y_test_cat.shape}")

In [ ]:
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Only normalize for validation and test (no augmentation)
val_datagen = ImageDataGenerator()

print("Data augmentation configured successfully")

## 3. Build the Model Architecture

In [ ]:
# Define custom layer: StandardizedConv2DWithOverride
class StandardizedConv2DWithOverride(Layer):
    """
    Custom Conv2D layer with standardization (Z-score normalization) 
    applied to the input before convolution.
    """
    def __init__(self, filters, kernel_size=3, strides=1, padding='same', 
                 activation=None, **kwargs):
        super(StandardizedConv2DWithOverride, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        self.padding = padding
        self.activation = activation

    def build(self, input_shape):
        self.conv = Conv2D(
            self.filters, 
            kernel_size=self.kernel_size,
            strides=self.strides, 
            padding=self.padding,
            activation=self.activation
        )
        super(StandardizedConv2DWithOverride, self).build(input_shape)

    def call(self, x):
        # Standardize input (Z-score normalization)
        mean = tf.reduce_mean(x, keepdims=True)
        std = tf.math.reduce_std(x, keepdims=True)
        x_standardized = (x - mean) / (std + 1e-7)  # Add small epsilon for numerical stability
        
        # Apply convolution
        return self.conv(x_standardized)

    def get_config(self):
        config = super(StandardizedConv2DWithOverride, self).get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size,
            'strides': self.strides,
            'padding': self.padding,
            'activation': self.activation
        })
        return config

print("Custom StandardizedConv2DWithOverride layer defined")

In [ ]:
# Build the custom CNN architecture with multi-scale feature fusion
def build_custom_cnn_model(input_shape=(48, 48, 3), num_classes=7):
    """
    Custom CNN architecture with multi-scale feature fusion paths.
    Combines features from different branches using concatenation.
    """
    # Input layer
    input_layer = Input(input_shape)
    
    # ===== FIRST SCALE PATH =====
    f1 = StandardizedConv2DWithOverride(32, kernel_size=3, strides=3, padding='same', activation='relu')(input_layer)
    f1 = BatchNormalization()(f1)
    f = Conv2D(32, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f = Conv2D(32, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = MaxPooling2D(2, 2)(f)
    f2 = Conv2D(32, kernel_size=1, strides=2, padding='same', activation='relu')(f)
    
    f1 = Conv2D(32, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f1 = Conv2D(32, kernel_size=3, strides=2, padding='same', activation='relu')(f1)
    
    f = concatenate([f, f1])
    f = BatchNormalization()(f)
    
    # ===== SECOND SCALE PATH =====
    f1 = Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f = Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu', name='BeforeFinal_Layer')(f)
    f = MaxPooling2D(2, 2)(f)
    f3 = Conv2D(32, kernel_size=1, strides=2, padding='same', activation='relu')(f)
    
    f1 = Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f1 = Conv2D(64, kernel_size=3, strides=2, padding='same', activation='relu')(f1)
    
    f = concatenate([f, f1, f2])
    f = BatchNormalization()(f)
    
    # ===== THIRD SCALE PATH =====
    f1 = Conv2D(128, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = Conv2D(128, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f = Conv2D(128, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = MaxPooling2D(2, 2)(f)
    f4 = Conv2D(32, kernel_size=1, strides=2, padding='same', activation='relu')(f)
    
    f1 = Conv2D(128, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f1 = Conv2D(128, kernel_size=3, strides=2, padding='same', activation='relu')(f1)
    f1 = BatchNormalization()(f1)
    
    f = concatenate([f, f1, f3])
    
    # ===== FOURTH SCALE PATH =====
    f1 = Conv2D(256, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = Conv2D(256, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f = Conv2D(256, kernel_size=3, strides=1, padding='same', activation='relu')(f)
    f = MaxPooling2D(2, 2)(f)
    
    f1 = Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu')(f1)
    f1 = Conv2D(256, kernel_size=3, strides=2, padding='same', activation='relu')(f1)
    f1 = BatchNormalization()(f1)
    
    f = concatenate([f, f1, f4])
    
    # ===== FINAL LAYERS =====
    f = Conv2D(512, kernel_size=3, strides=1, padding='same', activation='relu', name='Final_Layer')(f)
    f = BatchNormalization()(f)
    
    f = Flatten()(f)
    f = Dropout(rate=0.3)(f)
    f = Dense(1024, activation='relu')(f)
    f = Dropout(rate=0.32)(f)
    
    output_layer = Dense(num_classes, activation='softmax')(f)
    
    # Create model
    model = Model(inputs=[input_layer], outputs=[output_layer])
    return model

# Build the model
model = build_custom_cnn_model(input_shape=(48, 48, 3), num_classes=num_classes)

# Display model summary
model.summary()

In [ ]:
# Visualize the model architecture
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True, 
           rankdir='TB', dpi=100, expand_nested=True)
print("Model architecture saved to 'model_architecture.png'")

## 4. Compile the Model

In [ ]:
# Compile the model
optimizer = optimizers.Adam(learning_rate=0.0001)
loss_fn = keras.losses.CategoricalCrossentropy()

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

print("Model compiled successfully")
print(f"Optimizer: {optimizer}")
print(f"Loss function: {loss_fn}")
print(f"Metrics: accuracy")

## 5. Train the Model

In [ ]:
# Define callbacks
model_checkpoint = callbacks.ModelCheckpoint(
    '/saved_models/custom_cnn_best.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Training configuration
EPOCHS = 100
BATCH_SIZE = 32

print("Starting model training...")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Class weights: {class_weights_dict}")

In [ ]:
# Train the model with data augmentation
history = model.fit(
    train_datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val_cat),
    callbacks=[model_checkpoint, early_stopping, reduce_lr],
    class_weight=class_weights_dict,
    verbose=1
)

print("\nTraining completed!")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 6. Evaluate Model Performance

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Set Performance:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate predictions on test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\nDetailed Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")

# Per-class classification report
print("\nPer-Class Classification Report:")
class_names = [class_mapping[str(i)] for i in range(num_classes)]
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to 'confusion_matrix.png'")

## 7. Visualize Training History

In [ ]:
# Plot training and validation loss
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history saved to 'training_history.png'")

## 8. Make Predictions on Test Data

In [ ]:
# Function to visualize predictions
def visualize_predictions(X_test, y_test, y_pred_probs, num_samples=12):
    """Visualize sample predictions with confidence scores"""
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    # Randomly select samples
    indices = np.random.choice(len(X_test), num_samples, replace=False)
    
    for idx, ax in zip(indices, axes):
        # Handle both RGB and single channel images
        if X_test[idx].shape[-1] == 3:
            ax.imshow(X_test[idx])
        else:
            ax.imshow(X_test[idx].squeeze(), cmap='gray')
        
        true_label = class_names[y_test[idx]]
        pred_label = class_names[np.argmax(y_pred_probs[idx])]
        confidence = np.max(y_pred_probs[idx])
        
        color = 'green' if true_label == pred_label else 'red'
        title = f'True: {true_label}\nPred: {pred_label} ({confidence:.2%})'
        ax.set_title(title, color=color, fontsize=11, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Sample predictions saved to 'sample_predictions.png'")

# Visualize predictions
visualize_predictions(X_test, y_test, y_pred_probs, num_samples=12)

In [ ]:
# Save the final model
model.save('../saved_models/custom_cnn_final_model.h5')
print("Final model saved to '../saved_models/custom_cnn_final_model.h5'")

# Create a summary report
summary_report = {
    'model_name': 'Custom Multi-Scale CNN',
    'architecture': 'Custom CNN with feature fusion',
    'input_shape': (48, 48, 3),
    'num_classes': num_classes,
    'total_parameters': model.count_params(),
    'training_samples': len(X_train),
    'validation_samples': len(X_val),
    'test_samples': len(X_test),
    'epochs_trained': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
    'optimizer': 'Adam',
    'learning_rate': 0.0001,
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'training_completed': True,
    'timestamp': datetime.now().isoformat()
}

# Save summary report
with open('training_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
print(f"Model: {summary_report['model_name']}")
print(f"Total Parameters: {summary_report['total_parameters']:,}")
print(f"Test Accuracy: {summary_report['test_accuracy']:.4f}")
print(f"Test Loss: {summary_report['test_loss']:.4f}")
print(f"F1-Score: {summary_report['f1_score']:.4f}")
print(f"Training Summary saved to 'training_summary.json'")
print("="*50)